# 🎬 Kuku Promo Finder: Generalized & Strictly-Typed Pipeline
**Series:** *I Married a Broken Soldier. He was a Duke* (50 Episodes)  
**Track:** Find the Promos (Ads) Across Microdrama Series  
**Architecture:** Deterministic AST Parser + Generalized Sampler + Multi-Batch Extraction + Cross-Episode Synthesis + LLM Quality Critic  

---

### Key Architectural Tenets
1. **Generalized Parsing**: No hardcoded character names, scene headings, or ad-hoc file lists. Fully parameterized for any screenplay file or sample selection.
2. **Strict Type Formatting**: Native Python 3.13 strongly-typed dataclasses (`slots=True`, `frozen=True`), Enums, and runtime constraint checks.
3. **Integer Centisecond Math**: Converting all timestamps (`MM:SS.ss`) to integer centiseconds (`1 second = 100 cs`) to completely eradicate floating-point drift.
4. **Zero-Loss Data Model**: Preserves all 4 screenplay element types (`CAMERA`, `ACTION`, `SOUND FX`, `DIALOGUE`), along with off-screen indicators, delivery tones, and verbatim dialogue.


## ─── Phase 1: Environment Setup & Strictly-Typed AST Data Model ───
In this section, we:
- Define the **Strict Type Schema** (`ElementType`, `ScreenplayElement`, `Scene`, `EpisodeHeader`, `Episode`, `ValidationReport`).
- Ensure project folders exist (`input_scripts/`, `raw_llm_outputs/`) and configure `.gitignore`.
- Verify `os.environ["GEMINI_API_KEY"]` accessibility.
- Implement the **Generalized Parser Engine** capable of ingesting any screenplay file.
- Implement the **Generalized Inspector & Validator** to test any arbitrary sample of episodes (or the full corpus) against strict mechanical invariants.


In [3]:
from enum import Enum
from dataclasses import dataclass, field, asdict
from typing import Optional, List, Dict, Tuple, Literal, Any, Sequence
import re
import os
import json
from pathlib import Path
import pandas as pd

# ==============================================================================
# STRICT TYPE DEFINITIONS & SCHEMAS
# ==============================================================================

class ElementType(str, Enum):
    CAMERA = "CAMERA"
    ACTION = "ACTION"
    SOUND_FX = "SOUND FX"
    DIALOGUE = "DIALOGUE"


@dataclass(slots=True, frozen=True)
class DialogueMetadata:
    """Strongly typed breakdown of dialogue tags."""
    speaker: str
    modifier: Optional[str] = None   # e.g., 'O.S.', 'V.O.', 'Off-screen'
    delivery: Optional[str] = None   # e.g., 'Cold, Commanding', 'Grave, Alert'
    raw_tag: str = ""

    def __post_init__(self):
        if not self.speaker:
            raise ValueError("Speaker name cannot be empty in DialogueMetadata")


@dataclass(slots=True, frozen=True)
class ScreenplayElement:
    """Strictly typed atomic screenplay element conforming to integer centiseconds."""
    element_id: str                      # Format: EP{ep:02d}_EL{el:03d}
    episode_num: int
    scene_num: int
    element_type: ElementType
    start_raw: str                       # e.g., '00:11.30'
    end_raw: str                         # e.g., '00:13.10'
    start_cs: int                        # integer centiseconds, e.g. 1130
    end_cs: int                          # integer centiseconds, e.g. 1310
    duration_cs: int                     # end_cs - start_cs
    duration_seconds: float              # duration_cs / 100.0
    content: str                         # verbatim lines or visual action
    dialogue_meta: Optional[DialogueMetadata] = None
    raw_tag: str = ""

    def __post_init__(self):
        # Runtime Type & Value Assertions
        if not isinstance(self.element_type, ElementType):
            raise TypeError(f"Invalid element_type: {self.element_type}")
        if self.start_cs < 0:
            raise ValueError(f"start_cs must be non-negative: {self.start_cs}")
        if self.end_cs <= self.start_cs:
            raise ValueError(f"end_cs ({self.end_cs}) must be strictly greater than start_cs ({self.start_cs})")
        if self.duration_cs != (self.end_cs - self.start_cs):
            raise ValueError("duration_cs mismatch with (end_cs - start_cs)")
        if not self.content.strip():
            raise ValueError(f"Element {self.element_id} has empty content")


@dataclass(slots=True, frozen=True)
class Scene:
    """Strictly typed scene boundary containing atomic elements."""
    episode_num: int
    scene_num: int
    heading: str
    start_raw: str
    end_raw: str
    start_cs: int
    end_cs: int
    duration_cs: int
    elements: Tuple[ScreenplayElement, ...]

    def __post_init__(self):
        if self.end_cs <= self.start_cs:
            raise ValueError(f"Scene end_cs ({self.end_cs}) must exceed start_cs ({self.start_cs})")


@dataclass(slots=True, frozen=True)
class EpisodeHeader:
    """Strictly typed metadata header block."""
    episode_num: int
    raw_episode_title: str
    logline: str
    continuity_notes: str

    def __post_init__(self):
        if not self.logline.strip():
            raise ValueError(f"Episode {self.episode_num} logline cannot be empty")
        if not self.continuity_notes.strip():
            raise ValueError(f"Episode {self.episode_num} continuity notes cannot be empty")


@dataclass(slots=True, frozen=True)
class Episode:
    """Complete strictly typed episode containing headers, scenes, and elements."""
    header: EpisodeHeader
    scenes: Tuple[Scene, ...]
    elements: Tuple[ScreenplayElement, ...]
    total_elements: int
    total_duration_cs: int

    @property
    def episode_num(self) -> int:
        return self.header.episode_num

    @property
    def total_duration_seconds(self) -> float:
        return self.total_duration_cs / 100.0


@dataclass(slots=True, frozen=True)
class ValidationReport:
    """Deterministic validation audit for an episode."""
    episode_num: int
    filename: str
    headers_valid: bool
    scene_count: int
    camera_count: int
    action_count: int
    sound_fx_count: int
    dialogue_count: int
    total_elements: int
    timestamp_errors: int
    containment_errors: int
    is_valid: bool
    errors: Tuple[str, ...] = ()

print("[OK] Strict Type Definitions loaded successfully.")


[OK] Strict Type Definitions loaded successfully.


### 1.1 Directory Initialization & Confidentiality Controls
We verify workspace directories and establish [`.gitignore`](file:///c:/Users/Anirudh%20Singh/OneDrive/Documents/Kuku%20FM/.gitignore) to protect proprietary screenplays and API tokens.


In [4]:
WORKSPACE_DIR = Path.cwd()
INPUT_DIR = WORKSPACE_DIR / "input_scripts"
RAW_OUTPUTS_DIR = WORKSPACE_DIR / "raw_llm_outputs"

INPUT_DIR.mkdir(parents=True, exist_ok=True)
RAW_OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

# Strict confidentiality gitignore
gitignore_path = WORKSPACE_DIR / ".gitignore"
gitignore_content = """# Confidential Screenplay Corpus and Inputs
input_scripts/
*.txt

# Raw LLM Outputs and Intermediate Checkpoints
raw_llm_outputs/

# Environment Variables and Secrets
.env
*.env
*.key

# Python artifacts
__pycache__/
*.py[cod]
.venv/
venv/
"""
gitignore_path.write_text(gitignore_content, encoding="utf-8")

print(f"Workspace Path  : {WORKSPACE_DIR}")
print(f"input_scripts/  : {'[OK]' if INPUT_DIR.exists() else '[FAIL]'}")
print(f"raw_llm_outputs/: {'[OK]' if RAW_OUTPUTS_DIR.exists() else '[FAIL]'}")
print(f".gitignore      : {'[OK]' if gitignore_path.exists() else '[FAIL]'}")


Workspace Path  : c:\Users\Anirudh Singh\OneDrive\Documents\Kuku FM
input_scripts/  : [OK]
raw_llm_outputs/: [OK]
.gitignore      : [OK]


### 1.2 Verify `GEMINI_API_KEY` Accessibility


In [14]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types
from pydantic import BaseModel, Field

# 1. Load variables from models.env (change to load_dotenv() if your file is named .env)
load_dotenv("models.env")

# 2. Define api_key before checking it
api_key = os.getenv("GEMINI_API_KEY")

if api_key:
    masked = api_key[:4] + "..." + api_key[-4:] if len(api_key) > 8 else "***"
    print(f"[OK] GEMINI_API_KEY is detected: {masked}")
    
    # 3. Initialize client only after confirming the key exists
    client = genai.Client(api_key=api_key)
else:
    print("[WARN] GEMINI_API_KEY is not currently set in environment variables.")
    print("       Make sure 'models.env' is in the same folder as this notebook,")
    print('       or set it in your PowerShell session: $env:GEMINI_API_KEY="your-api-key"')


[OK] GEMINI_API_KEY is detected: AQ.A...ZORQ


### 1.3 Generalized Screenplay Parser Engine
This parser does not rely on hand-coded character names, hardcoded line counts, or brittle assumptions:
- Regex timestamps strictly validate against `^([0-9]{2}):([0-9]{2})\.([0-9]{2})$`.
- Integer centisecond conversion: $\text{MM} \times 6000 + \text{SS} \times 100 + \text{cs}$.
- Generalized tag decomposition handles single-paren and multi-paren modifiers (`(O.S.)`, `(Off-screen)`, `(V.O.)`, and tonal deliveries).
- Captures contiguous multi-line content for dialogue and visual descriptions.


In [6]:
TIMESTAMP_REGEX = re.compile(r"^([0-9]{2}):([0-9]{2})\.([0-9]{2})$")

def parse_centiseconds(ts_str: str) -> int:
    """Converts MM:SS.ss string to integer centiseconds (1s = 100cs)."""
    m = TIMESTAMP_REGEX.match(ts_str.strip())
    if not m:
        raise ValueError(f"Malformed timestamp format (expected MM:SS.ss): '{ts_str}'")
    mm, ss, cs = int(m.group(1)), int(m.group(2)), int(m.group(3))
    return mm * 6000 + ss * 100 + cs

def format_centiseconds(cs: int) -> str:
    """Converts integer centiseconds back to MM:SS.ss string."""
    mm = cs // 6000
    remainder = cs % 6000
    ss = remainder // 100
    centis = remainder % 100
    return f"{mm:02d}:{ss:02d}.{centis:02d}"

def decompose_tag(raw_tag_str: str) -> Tuple[ElementType, Optional[DialogueMetadata]]:
    """
    Generalized tag classification & decomposition:
    - CAMERA
    - ACTION
    - SOUND FX
    - DIALOGUE with any speaker, off-screen modifier, and delivery tone
    """
    clean_tag = raw_tag_str.strip()
    
    if clean_tag == "CAMERA":
        return ElementType.CAMERA, None
    if clean_tag == "ACTION":
        return ElementType.ACTION, None
    if clean_tag.startswith("SOUND FX"):
        return ElementType.SOUND_FX, None
        
    # Generalized Dialogue Tag Decomposition
    # Example 1: 'MATRIARCH (Cold, Commanding)'
    # Example 2: 'THE FORMER GROOM (Off-screen) (Dismissive)'
    # Example 3: 'JULIANA (V.O.) (Reflective, Vulnerable)'
    # Example 4: 'ALEXANDER'
    parens = re.findall(r"\((.*?)\)", clean_tag)
    first_paren_idx = clean_tag.find("(")
    speaker = clean_tag[:first_paren_idx].strip() if first_paren_idx != -1 else clean_tag.strip()
    
    modifier = None
    delivery = None
    
    if len(parens) == 1:
        p_val = parens[0].strip()
        if p_val.upper() in {"O.S.", "V.O.", "OFF-SCREEN", "OFF SCREEN"}:
            modifier = p_val
        else:
            delivery = p_val
    elif len(parens) >= 2:
        modifier = parens[0].strip()
        delivery = " | ".join(p.strip() for p in parens[1:])
        
    meta = DialogueMetadata(
        speaker=speaker,
        modifier=modifier,
        delivery=delivery,
        raw_tag=clean_tag
    )
    return ElementType.DIALOGUE, meta


def parse_screenplay_text(raw_text: str, fallback_episode_num: int) -> Episode:
    """
    Fully generalized, deterministic state-machine parser that converts raw screenplay text
    into a strictly-typed Episode object.
    """
    lines = raw_text.splitlines()
    
    # 1. Parse Episode Headers
    ep_num_match = re.search(r"^EPISODE NUMBER:\s*(?:Episode\s*)?(\d+)", raw_text, re.MULTILINE | re.IGNORECASE)
    ep_num = int(ep_num_match.group(1)) if ep_num_match else fallback_episode_num
    
    raw_title_match = re.search(r"^EPISODE NUMBER:\s*(.+)$", raw_text, re.MULTILINE)
    raw_title = raw_title_match.group(1).strip() if raw_title_match else f"Episode {ep_num}"
    
    logline_match = re.search(r"^EPISODE LOGLINE:\s*(.+)$", raw_text, re.MULTILINE)
    logline = logline_match.group(1).strip() if logline_match else ""
    
    notes_match = re.search(r"^CONTINUITY NOTES:\s*(.+)$", raw_text, re.MULTILINE)
    notes = notes_match.group(1).strip() if notes_match else ""
    
    header = EpisodeHeader(
        episode_num=ep_num,
        raw_episode_title=raw_title,
        logline=logline,
        continuity_notes=notes
    )
    
    # Locate body after '--- SCRIPT START ---'
    script_start_idx = raw_text.find("--- SCRIPT START ---")
    body_text = raw_text[script_start_idx + len("--- SCRIPT START ---"):] if script_start_idx != -1 else raw_text
    
    # 2. Parse Scenes
    scene_pattern = re.compile(
        r"^(SCENE\s+(?P<num>\d+))\s*\n"
        r"Scene heading:\s*(?P<heading>.+?)\s*\n"
        r"Scene start:\s*(?P<start>[0-9]{2}:[0-9]{2}\.[0-9]{2})\s*\n"
        r"Scene end:\s*(?P<end>[0-9]{2}:[0-9]{2}\.[0-9]{2})\s*\n"
        r"Actions and dialogue:",
        re.MULTILINE | re.IGNORECASE
    )
    
    scene_matches = list(scene_pattern.finditer(body_text))
    if not scene_matches:
        # Fallback single scene for edge cases
        scene_matches_spans = [(0, len(body_text), 1, "CONTINUOUS", "00:00.00", "00:00.00")]
    else:
        scene_matches_spans = []
        for i, sm in enumerate(scene_matches):
            start_pos = sm.end()
            end_pos = scene_matches[i + 1].start() if i + 1 < len(scene_matches) else len(body_text)
            scene_matches_spans.append((
                start_pos,
                end_pos,
                int(sm.group("num")),
                sm.group("heading").strip(),
                sm.group("start").strip(),
                sm.group("end").strip()
            ))
            
    # 3. Parse Elements within Each Scene
    element_pattern = re.compile(
        r"^\*\*(?P<tag>.+?):\*\*\s*\n"
        r"Start:\s*\((?P<start>[0-9]{2}:[0-9]{2}\.[0-9]{2})\)\s*\|\s*End:\s*\((?P<end>[0-9]{2}:[0-9]{2}\.[0-9]{2})\)\s*\n"
        r"(?P<content>.*?)(?=(?:\n\*\*.+?:\*\*\s*\nStart:|\nSCENE\s+\d+|\Z))",
        re.MULTILINE | re.DOTALL
    )
    
    parsed_scenes = []
    all_ep_elements = []
    element_counter = 0
    
    for s_start, s_end, sc_num, heading, sc_start_raw, sc_end_raw in scene_matches_spans:
        scene_body = body_text[s_start:s_end]
        sc_start_cs = parse_centiseconds(sc_start_raw)
        sc_end_cs = parse_centiseconds(sc_end_raw)
        
        scene_elements = []
        for em in element_pattern.finditer(scene_body):
            element_counter += 1
            el_id = f"EP{ep_num:02d}_EL{element_counter:03d}"
            
            raw_tag = em.group("tag").strip()
            start_raw = em.group("start").strip()
            end_raw = em.group("end").strip()
            content = em.group("content").strip()
            
            start_cs = parse_centiseconds(start_raw)
            end_cs = parse_centiseconds(end_raw)
            duration_cs = end_cs - start_cs
            
            el_type, dlg_meta = decompose_tag(raw_tag)
            
            element = ScreenplayElement(
                element_id=el_id,
                episode_num=ep_num,
                scene_num=sc_num,
                element_type=el_type,
                start_raw=start_raw,
                end_raw=end_raw,
                start_cs=start_cs,
                end_cs=end_cs,
                duration_cs=duration_cs,
                duration_seconds=duration_cs / 100.0,
                content=content,
                dialogue_meta=dlg_meta,
                raw_tag=raw_tag
            )
            scene_elements.append(element)
            all_ep_elements.append(element)
            
        scene = Scene(
            episode_num=ep_num,
            scene_num=sc_num,
            heading=heading,
            start_raw=sc_start_raw,
            end_raw=sc_end_raw,
            start_cs=sc_start_cs,
            end_cs=sc_end_cs,
            duration_cs=sc_end_cs - sc_start_cs,
            elements=tuple(scene_elements)
        )
        parsed_scenes.append(scene)
        
    total_dur_cs = sum(s.duration_cs for s in parsed_scenes) if parsed_scenes else (all_ep_elements[-1].end_cs if all_ep_elements else 0)
    
    return Episode(
        header=header,
        scenes=tuple(parsed_scenes),
        elements=tuple(all_ep_elements),
        total_elements=len(all_ep_elements),
        total_duration_cs=total_dur_cs
    )

print("[OK] Generalized Screenplay Parser Engine compiled.")


[OK] Generalized Screenplay Parser Engine compiled.


### 1.4 Generalized Mechanical Validator & Flexible Corpus Sampler
This engine validates any subset or the entire collection against mechanical invariants:
1. Complete header presence.
2. Chronological and non-zero scene boundaries.
3. Strict monotonicity ($0 \le \text{start\_cs} < \text{end\_cs}$).
4. Element containment within scene boundaries.
5. Exact element type distribution counts.


In [7]:
def validate_episode(ep: Episode, filename: str = "") -> ValidationReport:
    """Executes rigorous mechanical validation checks on an Episode instance."""
    errors = []
    
    headers_valid = bool(ep.header.logline.strip() and ep.header.continuity_notes.strip())
    if not headers_valid:
        errors.append(f"Missing header values in Ep {ep.episode_num}")
        
    camera_c = sum(1 for e in ep.elements if e.element_type == ElementType.CAMERA)
    action_c = sum(1 for e in ep.elements if e.element_type == ElementType.ACTION)
    sound_c = sum(1 for e in ep.elements if e.element_type == ElementType.SOUND_FX)
    dialogue_c = sum(1 for e in ep.elements if e.element_type == ElementType.DIALOGUE)
    
    time_errors = 0
    containment_errors = 0
    
    for sc in ep.scenes:
        if sc.start_cs >= sc.end_cs:
            time_errors += 1
            errors.append(f"Scene {sc.scene_num} invalid window: {sc.start_cs} >= {sc.end_cs}")
            
        for el in sc.elements:
            if el.start_cs >= el.end_cs:
                time_errors += 1
                errors.append(f"Element {el.element_id} timestamp monotonic failure: {el.start_cs} >= {el.end_cs}")
            # Element boundary should fall within scene window (with 1s leeway for audio tails)
            if el.start_cs < sc.start_cs or el.end_cs > (sc.end_cs + 100):
                containment_errors += 1
                
    is_valid = (len(errors) == 0 and time_errors == 0)
    
    return ValidationReport(
        episode_num=ep.episode_num,
        filename=filename,
        headers_valid=headers_valid,
        scene_count=len(ep.scenes),
        camera_count=camera_c,
        action_count=action_c,
        sound_fx_count=sound_c,
        dialogue_count=dialogue_c,
        total_elements=ep.total_elements,
        timestamp_errors=time_errors,
        containment_errors=containment_errors,
        is_valid=is_valid,
        errors=tuple(errors)
    )

def sample_episodes(
    files: Sequence[Path],
    sample_strategy: Literal["even_spread", "first_n", "last_n", "all"] = "even_spread",
    sample_size: int = 3
) -> List[Path]:
    """Generalized episode selection strategy for sampling."""
    if not files:
        return []
    if sample_strategy == "all" or sample_size >= len(files):
        return list(files)
    if sample_strategy == "first_n":
        return list(files[:sample_size])
    if sample_strategy == "last_n":
        return list(files[-sample_size:])
    if sample_strategy == "even_spread":
        # Evenly spaces sample across the beginning, middle, and end of the series
        step = (len(files) - 1) / (sample_size - 1) if sample_size > 1 else 0
        indices = [round(i * step) for i in range(sample_size)]
        return [files[i] for i in indices]
    return list(files[:sample_size])

print("[OK] Generalized Inspector & Mechanical Validator compiled.")


[OK] Generalized Inspector & Mechanical Validator compiled.


### 1.5 Execute Generalized Inspection Across Sampled Episodes
Select any sample strategy (e.g. `even_spread` across beginning, mid-series, and finale).  
The table below reports exact element distributions and validates **0 timestamp errors**.


In [8]:
# Get all episode files sorted
all_ep_files = sorted(INPUT_DIR.glob("ep*.txt"))
print(f"Total episode files in workspace: {len(all_ep_files)}")

# Select generalized sample (Beginning, Mid-Series, Climax Finale)
selected_samples = sample_episodes(all_ep_files, sample_strategy="even_spread", sample_size=3)
print("Inspecting representative sample files:", [p.name for p in selected_samples])

inspection_reports = []
sample_episodes_data = []

for p in selected_samples:
    text = p.read_text(encoding="utf-8")
    ep_num = int(re.search(r"\d+", p.stem).group(0))
    ep = parse_screenplay_text(text, fallback_episode_num=ep_num)
    sample_episodes_data.append(ep)
    
    report = validate_episode(ep, filename=p.name)
    inspection_reports.append({
        "Episode": f"Ep {ep.episode_num:02d}",
        "File": p.name,
        "Headers Valid": report.headers_valid,
        "Scenes": report.scene_count,
        "CAMERA": report.camera_count,
        "ACTION": report.action_count,
        "SOUND FX": report.sound_fx_count,
        "DIALOGUE": report.dialogue_count,
        "Total Elements": report.total_elements,
        "Timestamp Errors": report.timestamp_errors,
        "Status": "PASSED" if report.is_valid else "FAILED"
    })

df_inspection = pd.DataFrame(inspection_reports)
display(df_inspection)


Total episode files in workspace: 50
Inspecting representative sample files: ['ep01.txt', 'ep25.txt', 'ep50.txt']


,Episode,File,Headers Valid,Scenes,CAMERA,ACTION,SOUND FX,DIALOGUE,Total Elements,Timestamp Errors,Status
0,Ep 01,ep01.txt,True,1,50,7,1,37,95,0,PASSED
1,Ep 25,ep25.txt,True,1,44,9,0,42,95,0,PASSED
2,Ep 50,ep50.txt,True,2,19,6,1,14,40,0,PASSED


### 1.6 Generalized Verbatim Syntax & Strict Type Verification
Inspecting the strongly-typed AST fields (`speaker`, `modifier`, `delivery`, `duration_seconds`) dynamically pulled from the sampled episodes:


In [9]:
# Display real typed objects from the parsed sample episodes
sample_ep = sample_episodes_data[0]

print('=' * 70)
print(f'EPISODE {sample_ep.episode_num:02d} HEADER (Strictly Typed EpisodeHeader):')
print('=' * 70)
print(f'Title           : {sample_ep.header.raw_episode_title}')
print(f'Logline         : {sample_ep.header.logline[:110]}...')
print(f'Continuity Notes: {sample_ep.header.continuity_notes[:110]}...')

print('\n' + '=' * 70)
print('TYPED SCREENPLAY ELEMENTS ACROSS ALL 4 CATEGORIES:')
print('=' * 70)

categories_to_show = [ElementType.CAMERA, ElementType.ACTION, ElementType.SOUND_FX, ElementType.DIALOGUE]

for cat in categories_to_show:
    matching_el = next((e for e in sample_ep.elements if e.element_type == cat), None)
    if matching_el:
        print(f'\n[{cat.value.upper()} ELEMENT]')
        print(f'  ID        : {matching_el.element_id}')
        print(f'  Raw Tag   : **{matching_el.raw_tag}:**')
        print(f'  Window    : [{matching_el.start_raw} -> {matching_el.end_raw}] ({matching_el.duration_seconds:.2f}s | {matching_el.duration_cs} cs)')
        if matching_el.dialogue_meta:
            print(f'  Speaker   : {matching_el.dialogue_meta.speaker}')
            print(f'  Modifier  : {matching_el.dialogue_meta.modifier}')
            print(f'  Delivery  : {matching_el.dialogue_meta.delivery}')
        print(f'  Content   : {repr(matching_el.content[:120])}')


EPISODE 01 HEADER (Strictly Typed EpisodeHeader):
Title           : Episode 1
Logline         : Lady Juliana arrives at the cathedral in her late mother's bridal gown, only to suffer the ultimate betrayal w...
Continuity Notes: First episode of the series. Establishes Juliana's humiliation and abandonment by her betrothed, Gerald, and h...

TYPED SCREENPLAY ELEMENTS ACROSS ALL 4 CATEGORIES:

[CAMERA ELEMENT]
  ID        : EP01_EL001
  Raw Tag   : **CAMERA:**
  Window    : [00:00.00 -> 00:04.80] (4.80s | 480 cs)
  Content   : 'Symmetrical framing on the heavy church doors as they slowly part, pushing in slightly as Juliana stands framed in the l'

[ACTION ELEMENT]
  ID        : EP01_EL002
  Raw Tag   : **ACTION:**
  Window    : [00:01.00 -> 00:04.80] (3.80s | 380 cs)
  Content   : "Juliana pauses at the threshold, her expression bright with joyful anticipation, wearing her mother's vintage gown and a"

[SOUND FX ELEMENT]
  ID        : EP01_EL094
  Raw Tag   : **SOUND FX:**
  Window    :

### 1.7 Full 50-Episode Ingestion & AST Artifact Generation
We now parse all 50 episodes, mechanically validate every element, and export the canonical AST to `01_structured_episodes.json`.


In [10]:
print("Parsing all 50 episodes with the Generalized Strictly-Typed Engine...")

all_parsed_episodes = []
full_corpus_reports = []

for p in all_ep_files:
    text = p.read_text(encoding="utf-8")
    ep_num = int(re.search(r"\d+", p.stem).group(0))
    ep = parse_screenplay_text(text, fallback_episode_num=ep_num)
    all_parsed_episodes.append(ep)
    full_corpus_reports.append(validate_episode(ep, filename=p.name))

total_episodes = len(all_parsed_episodes)
total_elements = sum(ep.total_elements for ep in all_parsed_episodes)
total_scenes = sum(len(ep.scenes) for ep in all_parsed_episodes)
total_camera = sum(r.camera_count for r in full_corpus_reports)
total_action = sum(r.action_count for r in full_corpus_reports)
total_sound = sum(r.sound_fx_count for r in full_corpus_reports)
total_dialogue = sum(r.dialogue_count for r in full_corpus_reports)
total_time_errors = sum(r.timestamp_errors for r in full_corpus_reports)

print("=" * 60)
print("FULL CORPUS MECHANICAL VALIDATION RESULTS:")
print("=" * 60)
print(f"Total Episodes Processed  : {total_episodes:5d}  (Expected: 50)")
print(f"Total Scenes Detected     : {total_scenes:5d}")
print(f"Total Timed Elements      : {total_elements:5d}  (Blueprint: 3,725)")
print(f"  - CAMERA Elements       : {total_camera:5d}  (Blueprint: 1,754)")
print(f"  - ACTION Elements       : {total_action:5d}  (Blueprint:   372)")
print(f"  - SOUND FX Elements     : {total_sound:5d}  (Blueprint:   114)")
print(f"  - DIALOGUE Elements     : {total_dialogue:5d}  (Blueprint: 1,485)")
print(f"Total Timestamp Failures  : {total_time_errors:5d}  (Requirement: 0)")
print("=" * 60)

assert total_episodes == 50, f"Expected 50 episodes, got {total_episodes}"
assert total_time_errors == 0, "Discovered timestamp monotonicity errors!"

# Export AST to 01_structured_episodes.json
ast_export_path = WORKSPACE_DIR / "01_structured_episodes.json"

export_data = {
    "corpus_title": "I Married a Broken Soldier. He was a Duke",
    "total_episodes": total_episodes,
    "total_elements": total_elements,
    "element_distribution": {
        "CAMERA": total_camera,
        "ACTION": total_action,
        "SOUND_FX": total_sound,
        "DIALOGUE": total_dialogue
    },
    "episodes": [
        {
            "episode_num": ep.episode_num,
            "header": asdict(ep.header),
            "scenes": [
                {
                    "scene_num": sc.scene_num,
                    "heading": sc.heading,
                    "start_cs": sc.start_cs,
                    "end_cs": sc.end_cs,
                    "start_raw": sc.start_raw,
                    "end_raw": sc.end_raw,
                    "duration_cs": sc.duration_cs,
                    "elements": [asdict(e) for e in sc.elements]
                }
                for sc in ep.scenes
            ]
        }
        for ep in all_parsed_episodes
    ]
}

ast_export_path.write_text(json.dumps(export_data, indent=2), encoding="utf-8")
print(f"[OK] Exported canonical AST to {ast_export_path.name} ({ast_export_path.stat().st_size / 1024:.1f} KB)")


Parsing all 50 episodes with the Generalized Strictly-Typed Engine...
FULL CORPUS MECHANICAL VALIDATION RESULTS:
Total Episodes Processed  :    50  (Expected: 50)
Total Scenes Detected     :   100
Total Timed Elements      :  3727  (Blueprint: 3,725)
  - CAMERA Elements       :  1754  (Blueprint: 1,754)
  - ACTION Elements       :   372  (Blueprint:   372)
  - SOUND FX Elements     :   114  (Blueprint:   114)
  - DIALOGUE Elements     :  1487  (Blueprint: 1,485)
Total Timestamp Failures  :     0  (Requirement: 0)
[OK] Exported canonical AST to 01_structured_episodes.json (2474.3 KB)
